In [1]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec

In [4]:
# 1. Load Dataset Berita Detik.com

dataset_path = os.path.join("..", "01_crawling", "data_berita_detik.csv")
if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset tidak ditemukan di: {dataset_path}")

df = pd.read_csv(dataset_path)
print(f"Total Dokumen Berita : {len(df)} berita")
print(f"Distribusi Kelas     :\n{df['label'].value_counts().to_string()}\n")

Total Dokumen Berita : 200 berita
Distribusi Kelas     :
label
sport      100
finance    100



In [5]:
# 2. Kumpulan kata tidak baku / slang / singkatan yang umum di berita online
daftar_kata_tidak_baku = {
    "gak", "gk", "ga", "nggak", "ngga", "enggak", "kagak",
    "udah", "udh", "dah",
    "emang", "emg", "mmg",
    "banget", "bgt", "bngt",
    "aja", "aj",
    "gimana", "gmn",
    "gitu", "gtu",
    "kayak", "kyk",
    "biar", "byr",
    "doang", "dong", "dng",
    "nih", "neh",
    "tuh",
    "sih", "si",
    "yg", "yng",
    "dgn", "dg",
    "utk",
    "dlm",
    "tdk", "gk",
    "blm",
    "krn", "krna",
    "bs", "bsa",
    "jg", "jga",
    "sm", "sama2",
    "org",
    "trs", "trus", "terus2an",
    "lg", "lgi",
    "dr",
    "pd",
    "klo", "kalo",
    "ama",
    "bkn",
    "tp", "tpi",
    "dll",
    "dsb",
    "dst",
    "dkk",
    "wkwk", "wkwkwk", "haha", "hihi", "hehe",
    "btw", "fyi", "omg", "lol",
    "nyokap", "bokap",
    "cuy", "cuk", "bro", "sis", "gan",
    "gue", "gw", "gwa", "lo", "lu",
    "ngapain", "napa", "kenapa2",
    "mah", "atuh", "teh",
}

In [6]:
# 3. preprocessing
# Membuang angka, tanda baca, dan kata tidak baku

def preprocess_versi_2(teks):
    list_token = re.findall(r'\b[a-zA-Z]+\b', teks)
    clean_token = [kata for kata in list_token if kata.lower() not in daftar_kata_tidak_baku]
    return clean_token
    
data_token = [preprocess_versi_2(isi) for isi in df['isi_berita']]
total_kata = sum(len(kalimat) for kalimat in data_token)
kosakata_unik = len(set(kata for kalimat in data_token for kata in kalimat))

print(f"    - Total Kata Keseluruhan : {total_kata:,} kata")
print(f"    - Total Kosakata Unik    : {kosakata_unik:,} kata\n")

    - Total Kata Keseluruhan : 63,382 kata
    - Total Kosakata Unik    : 8,499 kata



In [7]:
# 4. Split Data

token_training, token_testing, label_training, label_testing = train_test_split(
    data_token,
    df['label'].values,
    train_size=0.8,
    test_size=0.2,
    stratify=df['label'].values
)

print(f"    - Data Training : {len(token_training)} dokumen")
print(f"    - Data Testing  : {len(token_testing)} dokumen")

df_split_training = pd.DataFrame({
    'teks_preprocessed': [' '.join(tokens) for tokens in token_training],
    'label': label_training
})
df_split_testing = pd.DataFrame({
    'teks_preprocessed': [' '.join(tokens) for tokens in token_testing],
    'label': label_testing
})

df_split_training.to_csv("split_v2_training.csv", index=False)
df_split_testing.to_csv("split_v2_testing.csv", index=False)

    - Data Training : 160 dokumen
    - Data Testing  : 40 dokumen


In [8]:
# 5. SKIP-GRAM

dimensi_vektor = 100
ukuran_jendela = 5

model_skipgram = Word2Vec(
    sentences=token_training,
    vector_size=dimensi_vektor,
    window=ukuran_jendela,
    min_count=1,
    sg=1,              
    epochs=30,         
    seed=42,
    workers=1
)

jumlah_kosakata_model = len(model_skipgram.wv)
print(f"    - Kosakata yang dipelajari model : {jumlah_kosakata_model:,} kata\n")


# Mean Pooling: mengubah vektor kata menjadi vektor dokumen
# rata-rata vektor dari seluruh kata dalam satu dokumen
def hitung_vektor_dokumen(list_token, model_skipgram, dimensi_vektor):
    vektor_kata_valid = [model_skipgram.wv[kata] for kata in list_token if kata in model_skipgram.wv]
    if len(vektor_kata_valid) > 0:
        return np.mean(vektor_kata_valid, axis=0)
    else:
        return np.zeros(dimensi_vektor)

fitur_training = np.array([hitung_vektor_dokumen(token, model_skipgram, dimensi_vektor) for token in token_training])
fitur_testing  = np.array([hitung_vektor_dokumen(token, model_skipgram, dimensi_vektor) for token in token_testing])

print(f"Hasil Embedding (Mean Pooling):")
print(f"    - Matriks Training : {fitur_training.shape} ({len(token_training)} dokumen x {dimensi_vektor} dimensi)")
print(f"    - Matriks Testing  : {fitur_testing.shape} ({len(token_testing)} dokumen x {dimensi_vektor} dimensi)\n")

    - Kosakata yang dipelajari model : 7,568 kata

Hasil Embedding (Mean Pooling):
    - Matriks Training : (160, 100) (160 dokumen x 100 dimensi)
    - Matriks Testing  : (40, 100) (40 dokumen x 100 dimensi)



In [9]:
# 6. Save to CSV
nama_kolom = [f"dim_{i+1}" for i in range(dimensi_vektor)]

df_training = pd.DataFrame(fitur_training, columns=nama_kolom)
df_training['label'] = label_training
df_training.to_csv("skipgram_v2_training.csv", index=False)

df_testing = pd.DataFrame(fitur_testing, columns=nama_kolom)
df_testing['label'] = label_testing
df_testing.to_csv("skipgram_v2_testing.csv", index=False)